In [ ]:
import numpy as np
import pandas as pd

import h5py
import anndata
from anndata._io.h5ad import read_elem

from scipy.sparse import csr_matrix

from tqdm.notebook import tqdm

from pprint import pprint
from IPython.display import display
from typing import List, Dict, Any, Literal, Tuple


import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
def detect_matrix_format(group) -> str:
    """
    Determine the matrix format based on the keys and structure of the given HDF5 group or dataset.

    This function analyzes the structure of an HDF5 group or dataset to identify the format
    of the matrix it represents. The function checks for common sparse matrix formats like 
    CSR, CSC, and COO, as well as dense matrix formats like NumPy arrays.

    Parameters:
    ----------
    group : h5py.Group or h5py.Dataset
        The HDF5 group or dataset to be analyzed. This could represent a sparse matrix 
        format (e.g., CSR, CSC, COO) or a dense matrix/array.

    Returns:
    -------
    str
        A string indicating the matrix format. Possible return values are:
        - "NumPy Array or Dense Matrix": Indicates that the group is a multi-dimensional 
          dataset, likely a dense matrix or NumPy array.
        - "1D Dataset": Indicates that the dataset is one-dimensional.
        - "CSR/CSC Matrix": Indicates that the group contains the necessary keys and 
          attributes to represent a CSR (Compressed Sparse Row) or CSC (Compressed Sparse Column) matrix.
        - "COO Matrix": Indicates that the group contains the necessary keys to represent 
          a COO (Coordinate) sparse matrix.
        - "Unknown Format": Indicates that the matrix format could not be determined based 
          on the group's structure.

    Notes:
    -----
    This function assumes that the input `group` is either a valid HDF5 group or dataset. 
    It does not perform extensive validation beyond checking the presence of specific keys 
    and attributes.
    """
    
    # If the group is an HDF5 dataset, check its shape and attributes.
    if isinstance(group, h5py.Dataset):
        # Check if it represents a multi-dimensional array (e.g., a dense matrix or NumPy array)
        if len(group.shape) >= 2:
            return "NumPy Array or Dense Matrix"
        else:
            return "1D Dataset"
    
    # Keys in the group
    keys = set(group.keys())
    
    # CSR or CSC Matrix Check
    if {'data', 'indices', 'indptr'}.issubset(keys):
        if 'shape' in group.attrs:
            return "CSR/CSC Matrix"
    
    # COO Matrix Check
    elif {'data', 'row', 'col'}.issubset(keys):
        return "COO Matrix"
    
    # Unknown format
    return "Unknown Format"

def generate_summary_report(data_dir: str) -> Dict[str, Any]:
    """
    Generate a summary report of the structure and content of an HDF5 file.

    This function analyzes the contents of an HDF5 file specified by `data_dir` and 
    generates a summary report. The report includes information about the groups and 
    datasets in the file, such as their keys, types, shapes, and formats. It specifically 
    handles the 'X' and 'layers' groups in a special manner to provide detailed information 
    about their structure, including the matrix format and shape.

    Parameters:
    ----------
    data_dir : str
        The path to the HDF5 file to be analyzed.

    Returns:
    -------
    Dict[str, Any]
        A dictionary containing the summary report of the HDF5 file. The structure of 
        the dictionary is as follows:
        - Keys are the names of the top-level groups or datasets.
        - Values are dictionaries that contain information about the keys and types of 
          groups, or the shape and dtype of datasets. For example:
            - For groups:
                {
                    "group_keys": List[str],       # List of keys within the group.
                    "group_types": List[type],     # List of types of the corresponding keys.
                    "format": str,                 # (For 'X' or 'layers' group) Matrix format.
                    "shape": tuple,                # (For 'X' or 'layers' group) Shape of the matrix.
                    "sublayer_info": dict          # (For 'layers' group) Information about sub-groups.
                }
            - For datasets:
                {
                    "shape": tuple,                # Shape of the dataset.
                    "dtype": numpy.dtype           # Data type of the dataset (if available).
                }

    Exceptions:
    ----------
    If an error occurs while processing the file, an error message is printed, and 
    an empty dictionary is returned.

    Notes:
    -----
    - The function is designed to be flexible and handle both dense (NumPy array) and 
      sparse (CSR, CSC, COO) matrix formats.
    - The shape of CSR/CSC matrices is derived from the 'indptr' and 'indices' datasets 
      if the 'shape' attribute is not available.
    - This function assumes that the 'X' group represents a matrix and processes it 
      accordingly. Similarly, the 'layers' group is expected to contain sub-groups 
      that represent different layers or matrices.
    """
    
    try:
        with h5py.File(data_dir, "r") as file:
            keys = file.keys()
            summary_report = {}
            
            for key in keys:
                group = file[key]
                
                if isinstance(group, h5py.Group):
                    group_keys = list(group.keys())
                    group_types = [type(group[k]) for k in group_keys]
                    summary_report[key] = {
                        "group_keys": group_keys,
                        "group_types": group_types,
                    }
                    
                    # Process the 'X' group
                    if key == 'X':
                        matrix_format = detect_matrix_format(group)
                        summary_report[key]["format"] = matrix_format
                        
                        # Report the shape directly for 'X'
                        if matrix_format in ["CSR/CSC Matrix", "COO Matrix"]:
                            if 'shape' in group.attrs:
                                summary_report[key]["shape"] = group.attrs['shape']
                            else:
                                # Fallback shape calculation (try to avoid if possible)
                                summary_report[key]["shape"] = (
                                    group["indptr"].shape[0] - 1,
                                    max(group["indices"]) + 1
                                )
                        elif matrix_format == "NumPy Array or Dense Matrix":
                            summary_report[key]["shape"] = group.shape  # Directly use group.shape

                    # Process the 'layers' group
                    if key == 'layers': 
                        layer_dict = {}
                        for k in file["layers"].keys():
                            sub_group = file[key][k]
                            
                            matrix_format = detect_matrix_format(sub_group)
                            layer_dict[k] = {
                                "format": matrix_format,
                            }
                            
                            # Directly use the shape if available
                            if 'shape' in sub_group.attrs:
                                layer_dict[k]["shape"] = sub_group.attrs['shape']
                            else:
                                if matrix_format == "NumPy Array or Dense Matrix":
                                    layer_dict[k]["shape"] = sub_group.shape
                                elif matrix_format == "CSR/CSC Matrix":
                                    # Only calculate shape manually if absolutely necessary
                                    layer_dict[k]["shape"] = (
                                        sub_group["indptr"].shape[0] - 1,
                                        max(sub_group["indices"]) + 1
                                    )
                        
                        summary_report[key]["sublayer_info"] = layer_dict
                    
                elif isinstance(group, h5py.Dataset):
                    dtype = group.dtype if hasattr(group, "dtype") else None
                    summary_report[key] = {"shape": group.shape, "dtype": dtype}
        
        return summary_report
    
    except Exception as e:
        print(f"Error processing file: {e}")
        return {}


In [ ]:
#data_dir = '/nfs/team298/ar32/repos/H5py_anndata_checker/dummy_data/test1.h5ad'
#data_dir = '/nfs/team298/ar32/repos/H5py_anndata_checker/dummy_data/test2.h5ad'
data_dir = '/nfs/team298/ar32/projects/IA/data/sc_objects/IA_round_3_exploration/major_lineages/adata_haem_with_leiden.h5ad'

In [ ]:
def generate_anndata_report(
    data_dir: str,
):
    """
    Generate and print a detailed report of the contents of an HDF5 file assumed to represent an AnnData object.

    This function analyzes the structure of an HDF5 file at the specified `data_dir` by invoking the 
    `generate_summary_report` function. It prints out a formatted report that includes information about 
    the file's groups, datasets, and specific matrix formats, such as the main data matrix ('X') and any 
    additional layers. The report highlights the structure, data types, shapes, and number of cells and features 
    in these matrices, which are common attributes in AnnData objects used in single-cell RNA sequencing (scRNA-seq) analysis.

    Parameters:
    ----------
    data_dir : str
        The path to the HDF5 file to be analyzed.

    Returns:
    -------
    None
        This function does not return a value. Instead, it prints the report directly to the console.

    Notes:
    -----
    - The function is designed to handle HDF5 files structured similarly to AnnData objects, which often contain 
      groups like 'X' (the main data matrix) and 'layers' (additional matrices). The 'X' group is typically 
      a matrix where rows correspond to cells and columns correspond to features (e.g., genes).
    - The function prints detailed information including:
        - The keys and types of each top-level group.
        - For the 'X' group, it prints the matrix format, shape, and the number of cells and features.
        - For the 'layers' group, it prints information for each sub-layer, including matrix format, shape, 
          and the number of cells and features.
    - The output is formatted with bold section titles and is intended for easy readability in a console environment.

    Example:
    -------
    To generate a report for an AnnData HDF5 file:

    >>> generate_anndata_report("path/to/anndata_file.h5")

    This would print a detailed report about the contents and structure of the HDF5 file.
    """
    
    report = generate_summary_report(data_dir)
    print(f"\033[1mAnndata object checking: {data_dir}\033[0m\n")

    # Print the summary report
    for key, value in report.items():
        print(f"\033[1mPartition: {key}\033[0m\n")

        if "group_keys" in value:
            group_info = {
                "Group Keys": value['group_keys'],
                "Group Types": value['group_types']
            }
            pprint(group_info, indent=4, width=80, compact=True)

            if key == "X":
                matrix_info = {
                    "Matrix Format": value['format'],
                    "Shape": value.get("shape"),
                }

                # Extract and add number of cells and features from shape
                if "shape" in value:
                    num_cells = value["shape"][0]
                    num_features = value["shape"][1]
                    matrix_info["Number of cells"] = num_cells
                    matrix_info["Number of features"] = num_features

                pprint(matrix_info, indent=4, width=80, compact=True)

            elif key == "layers":
                print("\nSub layer information:\n")
                for k, v in value["sublayer_info"].items():
                    print(f"{k}:")
                    layer_info = {
                        "Matrix Format": v['format'],
                    }

                    # Handle shape and extract number of cells and features if shape is present
                    if "shape" in v:
                        layer_info["Shape"] = v["shape"]
                        num_cells = v["shape"][0]
                        num_features = v["shape"][1]
                        layer_info["Number of cells"] = num_cells
                        layer_info["Number of features"] = num_features

                    if "Cell_num" in v:
                        layer_info["Number of cells"] = v["Cell_num"]
                    if "Feat_num" in v:
                        layer_info["Number of features"] = v["Feat_num"]

                    pprint(layer_info, indent=6, width=80, compact=True)
                    print("")  # Empty line for readability

        else:
            dataset_info = {
                "Shape": value['shape'],
                "Dtype": value['dtype']
            }
            pprint(dataset_info, indent=4, width=80, compact=True)

        print("---------------------------\n")

In [ ]:
generate_anndata_report(data_dir)

In [ ]:
def inspect_column_categories(
    data_dir: str,
    dataframe: Literal['obs', 'var'],
    columns: List[str],
) -> pd.DataFrame:
    """
    Inspect and retrieve categories or unique values from specified columns within an AnnData-like HDF5 file.

    This function examines specific columns in a given dataframe group (either 'obs' or 'var') 
    within an HDF5 file. It decodes the categories for categorical columns or gathers unique values 
    for non-categorical columns, then compiles the results into a pandas DataFrame for easy inspection.

    Parameters:
    ----------
    data_dir : str
        The path to the HDF5 file containing the AnnData-like structure.
        
    dataframe : Literal['obs', 'var']
        The name of the dataframe group within the HDF5 file to inspect. Typically, 'obs' refers to 
        observations (cells), and 'var' refers to variables (features/genes).
        
    columns : List[str]
        A list of column names to inspect within the specified dataframe. These columns may contain 
        categorical data or other types of data for which unique values will be retrieved.

    Returns:
    -------
    pd.DataFrame
        A DataFrame where each column corresponds to one of the inspected columns from the HDF5 file. 
        The DataFrame contains either decoded category values (for categorical columns) or unique values 
        (for non-categorical columns). The DataFrame columns are renamed to indicate that they contain 
        unique values.

    Raises:
    ------
    ValueError:
        If the specified dataframe ('obs' or 'var') is not found in the HDF5 file, a ValueError is raised.

    Notes:
    -----
    - If a column is not found in the specified dataframe group, the corresponding DataFrame column 
      will contain "Column not found" as its value.
    - The resulting DataFrame is padded with empty strings to ensure all columns have the same length, 
      which is equal to the maximum number of unique values found across all specified columns.
    - This function assumes that categorical data in the HDF5 file is stored with a 'categories' attribute, 
      and non-categorical data is stored as arrays from which unique values can be derived.

    Example:
    -------
    To inspect the unique values or categories of certain columns in the 'obs' group of an HDF5 file:

    >>> df = inspect_column_categories("path/to/anndata_file.h5", "obs", ["cell_type", "condition"])
    >>> print(df)

    This would output a DataFrame showing the unique values or decoded categories for the specified columns.
    """
    
    # Initialize an empty dictionary to store decoded categories or unique values
    decoded_data: Dict[str, List[Any]] = {col: [] for col in columns}
    
    with h5py.File(data_dir, "r") as file:
        # Check if the specified dataframe exists in the file
        if dataframe not in file:
            raise ValueError(f"DataFrame type '{dataframe}' not found in file.")
        
        df_group = file[dataframe]
        
        # Iterate over columns and decode categories or gather unique values
        for col in columns:
            if col in df_group:
                data = df_group[col]
                
                if "categories" in data:
                    # Decode category values if the column has categories
                    decoded_data[col] = [value.decode() for value in data["categories"]]
                else:
                    # For non-categorical columns, gather unique values
                    # Note: Adjust depending on actual data storage
                    decoded_data[col] = list(set(data))
            else:
                # Handle the case where the column is not found
                decoded_data[col] = ["Column not found"] 

    # Find the maximum length among all categories or unique values
    max_length = max(len(decoded_data[col]) for col in columns)
    
    # Pad the lists in the dictionary with empty strings if needed
    for col in columns:
        if len(decoded_data[col]) < max_length:
            decoded_data[col] += [""] * (max_length - len(decoded_data[col]))
    
    # Create a DataFrame from the decoded data
    df = pd.DataFrame(decoded_data)
    
    # Update column names with ' unique values'
    df.columns = [f"{col} unique values" for col in columns]
    
    return df


In [ ]:
dataframe = 'obs'
columns = [
    'LVL0', 
    'LVL1',
    #'concatenated_integration_covariates',
    'organ'
    ]

print(
    f"\033[1mDataFrame output to see all unique values for each column of interest in {dataframe}:\033[0m\n"
)
inspect_column_categories(data_dir, dataframe, columns)

In [ ]:
dataframe = 'var' # choose obs or var 
columns = [
    'hgnc',
    'intersection'
    ]

# Display the DataFrame
print(
    f"\033[1mDataFrame output to see all unique values for each column of interest in {dataframe}:\033[0m\n"
)
inspect_column_categories(data_dir, dataframe, columns)

In [ ]:
dataframe = 'var' # choose obs or var 
columns = [
    'highly_variable_intersection'
    ]

# Display the DataFrame
print(
    f"\033[1mDataFrame output to see all unique values for each column of interest in {dataframe}:\033[0m\n"
)
inspect_column_categories(data_dir, dataframe, columns)

In [ ]:
def extract_dataframe(
    file,
    dataframe: Literal['obs', 'var'],
    filter_dict: Dict[str, List[str]],
    additional_cols: List[str],
    filter_method: Literal['intersection', 'union'] = 'intersection'
):
    """
    Extract and filter a dataframe from an HDF5 file based on specified criteria.

    This function extracts a dataframe ('obs' or 'var') from an HDF5 file, filters it based 
    on criteria specified in `filter_dict`, and optionally adds additional columns. The filtering 
    can be performed using either an intersection or union method, depending on the `filter_method` parameter.

    Parameters:
    ----------
    file : h5py.File
        An open HDF5 file handle from which to extract the data.
    
    dataframe : Literal['obs', 'var']
        The name of the dataframe to extract from the HDF5 file. Typically, 'obs' refers to 
        observations (cells) and 'var' refers to variables (features/genes).
    
    filter_dict : Dict[str, List[str]]
        A dictionary where keys are column names in the dataframe and values are lists of 
        filter values. Only rows where the column values match the filter values will be retained.

    additional_cols : List[str]
        A list of additional column names to extract from the dataframe and include in the 
        output DataFrame, after filtering.

    filter_method : Literal['intersection', 'union'], default='intersection'
        The method to use when combining filters:
        - 'intersection': Only include rows that match all filter criteria (AND logic).
        - 'union': Include rows that match any filter criteria (OR logic).

    Returns:
    -------
    pd.DataFrame
        A pandas DataFrame containing the filtered and optionally extended data. The DataFrame 
        includes the filtered columns specified in `filter_dict` and any additional columns 
        specified in `additional_cols`. The original index values and positions are also included.

    Raises:
    ------
    ValueError:
        - If the specified `dataframe` is not found in the HDF5 file.
        - If any values in `filter_dict` are not found in the corresponding columns.
        - If an invalid `filter_method` is provided.

    Notes:
    -----
    - The original index values and positions are included in the output DataFrame to maintain 
      traceability back to the original data.
    - The function assumes that the `_index` column is encoded as a byte string, which is decoded 
      to UTF-8 for processing.
    - The output DataFrame columns are reordered according to the `column-order` attribute in 
      the HDF5 file, if available.

    Example:
    -------
    To extract and filter data from the 'obs' dataframe based on certain criteria and include 
    additional columns:

    >>> df = extract_dataframe(
            file=h5py.File("path/to/anndata_file.h5", "r"),
            dataframe="obs",
            filter_dict={"cell_type": ["B cell", "T cell"]},
            additional_cols=["age", "condition"],
            filter_method="intersection"
        )
    >>> print(df)

    This would output a DataFrame with only the cells of type "B cell" or "T cell", and includes 
    the 'age' and 'condition' columns.
    """
    
    # Retrieve and decode index
    original_index_values = np.vectorize(lambda x: x.decode("utf-8"))(
        np.array(file[dataframe]["_index"], dtype=object)
    )
    
    # Create a DataFrame with both original index values and their positions
    original_index_df = pd.DataFrame({
        "Original_index_value": original_index_values,
        "Original_index_position": np.arange(len(original_index_values))
    }, index=original_index_values)

    # Initialize a list to keep DataFrames for filtering
    filtered_dfs = []

    # Apply filtering for each column specified in filter_dict
    for filter_column, filter_values in filter_dict.items():
        col_data = pd.DataFrame(read_elem(file[dataframe][filter_column])).astype(str)
        col_data = col_data[col_data.iloc[:, 0].isin(filter_values)]
        missing_values = set(filter_values) - set(col_data.iloc[:, 0])

        if missing_values:
            raise ValueError(f"Missing values in filter_column '{filter_column}': {missing_values}")

        col_data.rename(columns={0: filter_column}, inplace=True)
        col_data.index = original_index_values[col_data.index]
        
        # Add the original index value and position columns to col_data
        col_data.insert(0, "Original_index_value", col_data.index)
        col_data.insert(1, "Original_index_position", original_index_df.loc[col_data.index, "Original_index_position"])

        filtered_dfs.append(col_data)

    # Combine filtered DataFrames based on filter_method
    if filter_method == 'intersection':
        # Intersect all filtered DataFrames
        common_index = set(filtered_dfs[0].index)
        for df in filtered_dfs[1:]:
            common_index &= set(df.index)
        common_index = list(common_index)
    elif filter_method == 'union':
        # Union of all filtered DataFrames
        common_index = set()
        for df in filtered_dfs:
            common_index |= set(df.index)
        common_index = list(common_index)
    else:
        raise ValueError("Invalid filter_method. Choose either 'intersection' or 'union'.")

    # Combine columns into a single DataFrame
    combined_cols = {df.columns[2]: df[df.index.isin(common_index)] for df in filtered_dfs}

    # Add additional columns
    if additional_cols:
        for col in additional_cols:
            col_data = pd.DataFrame(read_elem(file[dataframe][col]))
            col_data.rename(columns={0: col}, inplace=True)
            col_data.index = original_index_values
            col_data = col_data[col_data.index.isin(common_index)]
            combined_cols[col] = col_data
    else:
        df = pd.DataFrame(read_elem(file[dataframe]))
        df.index = original_index_values
        df = df[df.index.isin(common_index)]
        df["Original_index_value"] = df.index
        df["Original_index_position"] = original_index_df.loc[df.index, "Original_index_position"]
        for col in list(df.columns):
            combined_cols[col] = df[[col]]

    # Final DataFrame assembly
    out_df = pd.concat(combined_cols.values(), axis=1)
    original_col_order = ["Original_index_position", "Original_index_value"] + list(file[dataframe].attrs.get("column-order", []))
    original_col_order = [item for item in original_col_order if item in out_df.columns]
    out_df = out_df.loc[:, ~out_df.columns.duplicated()]
    out_df = out_df.filter(items=original_col_order)
    out_df.reset_index(drop=True, inplace=True)
    
    return out_df

def create_dataframe_subset(
    data_dir: str,
    dataframe: Literal['obs', 'var'],
    filter_dict: Dict[str, List[str]],  # Dictionary to specify filter columns and their values
    additional_cols: List[str],
    filter_method: Literal['intersection', 'union'] = 'intersection'  # Method to apply filtering
) -> pd.DataFrame:
    """
    Create a subset of a dataframe from an HDF5 file based on specified filtering criteria.

    This function loads an HDF5 file, extracts a specific dataframe ('obs' or 'var'), applies filtering 
    based on the criteria provided in `filter_dict`, and optionally includes additional columns. 
    The filtering can be performed using either an intersection or union method.

    Parameters:
    ----------
    data_dir : str
        The path to the HDF5 file containing the AnnData-like structure.
    
    dataframe : Literal['obs', 'var']
        The name of the dataframe to subset from the HDF5 file. Typically, 'obs' refers to 
        observations (cells) and 'var' refers to variables (features/genes).
    
    filter_dict : Dict[str, List[str]]
        A dictionary where keys are column names in the dataframe and values are lists of 
        filter values. Only rows where the column values match the filter values will be retained.
    
    additional_cols : List[str]
        A list of additional column names to extract from the dataframe and include in the 
        subset after filtering.

    filter_method : Literal['intersection', 'union'], default='intersection'
        The method to use when combining the filter criteria:
        - 'intersection': Only include rows that match all filter criteria (AND logic).
        - 'union': Include rows that match any filter criteria (OR logic).

    Returns:
    -------
    pd.DataFrame
        A pandas DataFrame containing the filtered subset of data. The DataFrame includes 
        the filtered columns specified in `filter_dict` and any additional columns specified 
        in `additional_cols`.

    Example:
    -------
    To create a subset of the 'obs' dataframe with specific filter criteria and include additional columns:

    >>> subset_df = create_dataframe_subset(
            data_dir="path/to/anndata_file.h5",
            dataframe="obs",
            filter_dict={"cell_type": ["B cell", "T cell"], "condition": ["treated"]},
            additional_cols=["age", "gender"],
            filter_method="intersection"
        )
    >>> print(subset_df)

    This would extract a subset of the 'obs' dataframe, filtering for rows where 'cell_type' 
    is either "B cell" or "T cell", and 'condition' is "treated", while also including the 
    'age' and 'gender' columns.
    """
    
    
    with h5py.File(data_dir, "r") as file:
        
        subset_dataframe = extract_dataframe(file, dataframe, filter_dict, additional_cols, filter_method)
        
    return subset_dataframe



In [ ]:
dataframe = 'obs'

filter_dict = {
    
    'LVL0' : ['Haematopoeitic_lineage'],
    #'anno_LVL1' : ['epithelial']
    'organ' : ['Yolk_sac ']
}

additional_cols_keep = [
    'LVL1',
    'LVL2',
    'LVL3'
    ]

filter_method = 'intersection'
#filter_method = 'union'


print(
        f"\033[1mDataFrame output of {dataframe} subset by columns {list(filter_dict.keys())} by {filter_method} for values of interest:\033[0m\n"
    )
create_dataframe_subset(data_dir, dataframe, filter_dict, additional_cols_keep, filter_method)

In [ ]:
dataframe = 'obs'

filter_dict = {
    
    'LVL0' : ['Haematopoeitic_lineage'],
    #'anno_LVL1' : ['epithelial']
    'organ' : ['Yolk_sac ']
}

additional_cols_keep = [
    ]

filter_method = 'intersection'
#filter_method = 'union'


print(
        f"\033[1mDataFrame output of {dataframe} subset by columns {list(filter_dict.keys())} by {filter_method} for values of interest:\033[0m\n"
    )
create_dataframe_subset(data_dir, dataframe, filter_dict, additional_cols_keep, filter_method)

In [ ]:
dataframe = 'var'

filter_dict = {
    
    'intersection' : ['1'],
    'highly_variable_reason' : ['hvg'],
    'highly_variable_intersection' : ['False']
}

additional_cols_keep = []

filter_method = 'intersection'
#filter_method = 'union'


print(
        f"\033[1mDataFrame output of {dataframe} subset by columns {list(filter_dict.keys())} by {filter_method} for values of interest:\033[0m\n"
    )
create_dataframe_subset(data_dir, dataframe, filter_dict, additional_cols_keep, filter_method)

In [ ]:
def sparse_grab_filtered_values(
    rows_to_load: List[int],
    cols_to_load: List[int],
    data_dset: np.ndarray,
    indices_dset: np.ndarray,
    indptr_dset: np.ndarray,
    description: str,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Extract data for specific rows and columns from a sparse matrix in Compressed Sparse Row (CSR) or 
    Compressed Sparse Column (CSC) format.

    This function filters and retrieves the data from a sparse matrix stored in CSR or CSC format, 
    based on the specified rows and columns. The data, indices, and indptr arrays corresponding to 
    the filtered subset are returned, allowing for efficient subsetting of the matrix.

    Parameters:
    ----------
    rows_to_load : List[int]
        A list of row indices to extract from the sparse matrix. The indices should correspond 
        to the rows of interest in the original matrix.
    
    cols_to_load : List[int]
        A list of column indices to extract from the sparse matrix. The indices should correspond 
        to the columns of interest in the original matrix.
    
    data_dset : np.ndarray
        The 'data' array from the sparse matrix dataset, which contains the non-zero values of 
        the matrix.
    
    indices_dset : np.ndarray
        The 'indices' array from the sparse matrix dataset, which contains the column (for CSR) 
        or row (for CSC) indices corresponding to each non-zero value in the 'data' array.
    
    indptr_dset : np.ndarray
        The 'indptr' array from the sparse matrix dataset, which defines the boundaries of the 
        rows (for CSR) or columns (for CSC) in the 'data' and 'indices' arrays.
    
    description : str
        A descriptive string for the operation, which is used for display purposes in progress 
        indicators.
    
    Returns:
    -------
    Tuple[np.ndarray, np.ndarray, np.ndarray]
        - selected_data: A numpy array containing the filtered non-zero values from the 'data' array.
        - selected_indices: A numpy array containing the filtered column (for CSR) or row (for CSC) 
          indices corresponding to the 'selected_data'.
        - selected_indptr: A numpy array defining the boundaries of the filtered rows (for CSR) 
          or columns (for CSC) in the 'selected_data' and 'selected_indices' arrays.

    Example:
    -------
    To extract specific rows and columns from a CSR sparse matrix:

    >>> rows = [0, 2, 4]
    >>> cols = [1, 3, 5]
    >>> data, indices, indptr = sparse_grab_filtered_values(
            rows_to_load=rows,
            cols_to_load=cols,
            data_dset=csr_matrix.data,
            indices_dset=csr_matrix.indices,
            indptr_dset=csr_matrix.indptr,
            description="Subsetting CSR Matrix"
        )

    This would return the filtered non-zero values, indices, and indptr arrays for the specified 
    rows and columns in the CSR matrix.
    """
    selected_data = []
    selected_indices = []
    selected_indptr = [0]

    # Mapping from original to filtered indices
    row_map = {orig_idx: new_idx for new_idx, orig_idx in enumerate(rows_to_load)}
    col_map = {orig_idx: new_idx for new_idx, orig_idx in enumerate(cols_to_load)}

    # Process each row
    for row_idx in tqdm(
        rows_to_load,
        desc=f"Processing Rows and Columns for {description}",
        unit="row",
        position=0,
        leave=True,
    ):
        start_idx = indptr_dset[row_idx]
        end_idx = indptr_dset[row_idx + 1]

        # Filter the indices and data for the columns we are interested in
        row_indices = indices_dset[start_idx:end_idx]
        row_data = data_dset[start_idx:end_idx]

        # Adjust indices to match filtered columns
        filtered_indices = [col_map.get(idx, -1) for idx in row_indices if idx in col_map]
        filtered_data = [row_data[i] for i in range(len(row_indices)) if row_indices[i] in col_map]

        # Add the filtered data to the list
        selected_data.extend(filtered_data)
        selected_indices.extend(filtered_indices)
        selected_indptr.append(selected_indptr[-1] + len(filtered_data))

    selected_data = np.array(selected_data)
    selected_indices = np.array(selected_indices)
    selected_indptr = np.array(selected_indptr)

    return selected_data, selected_indices, selected_indptr






def create_anndata_subset(
    data_dir: str,
    obs_filter_dict: Dict[str, List[str]],
    obs_additional_cols_keep: List[str],
    obs_filter_method: Literal['intersection', 'union'],
    var_filter_dict: Dict[str, List[str]],
    var_additional_cols_keep: List[str],
    var_filter_method: Literal['intersection', 'union'],
    filter_layers: List[str],
    filter_obsm: List[str],
    filter_obsp: List[str],
    filter_varm: List[str],
    filter_varp: List[str],
    filter_uns: List[str],
    keep_layers: bool = False,
    keep_obsm: bool = False,
    keep_obsp: bool = False,
    keep_varm: bool = False,
    keep_varp: bool = False,
    keep_uns: bool = False,
):
    
    with h5py.File(data_dir, "r") as file:
        
        if obs_filter_dict:
            obs_dataframe = extract_dataframe(file, "obs", obs_filter_dict, obs_additional_cols_keep, obs_filter_method)
            
            # List of row positions to load
            obs_rows_to_load = obs_dataframe["Original_index_position"].values
            #display(obs_rows_to_load)
        
        if var_filter_dict:
            var_dataframe = extract_dataframe(file, "var", var_filter_dict, var_additional_cols_keep, var_filter_method)
        
            # List of row positions to load
            var_cols_to_load = var_dataframe["Original_index_position"].values
            #display(var_rows_to_load)
            
        
        matrix_format = detect_matrix_format(file["X"])
        
        if matrix_format in ["CSR/CSC Matrix", "COO Matrix"]: # first test with csr, check later for csc and coo
            
            # Assign variables to query
            data_dset = file["X"]["data"]
            indices_dset = file["X"]["indices"]
            indptr_dset = file["X"]["indptr"]
            
            
            # Create the subset matrix
            print(
                        "Look up cells of interest:  \U0001F50D",
                        flush=True,
                    )
            
            # Extract filtered data, indices, and indptr
            filtered_data, filtered_indices, filtered_indptr = sparse_grab_filtered_values(
                obs_rows_to_load,
                var_cols_to_load,
                data_dset,
                indices_dset,
                indptr_dset,
                "filtered data"
            )

            num_filtered_rows = len(obs_rows_to_load)
            num_filtered_cols = len(var_cols_to_load)

            # Create the subset matrix
            print(
                        "Constructing data into csr_matrix format:  \U0001F527",
                        flush=True,
                    )
            subset_matrix = csr_matrix(
                (filtered_data, filtered_indices, filtered_indptr),
                shape=(num_filtered_rows, num_filtered_cols),
                dtype=data_dset.dtype
            )
            print("Construction complete \u2705")
        
        
        # requires testing
        elif matrix_format == "NumPy Array or Dense Matrix":
            subset_matrix = file["X"][obs_rows_to_load, :][:, var_cols_to_load]
        
        obs_dataframe.index = obs_dataframe['Original_index_value'].copy()
        obs_dataframe.index.name = None
        
        var_dataframe.index = var_dataframe['Original_index_value'].copy()
        var_dataframe.index.name = None
        
        if keep_layers:
            if not filter_layers:
                layers = {}
                for x in file["layers"].keys():
                    # Assign variables to query
                    data_dset = file["layers"][x]["data"]
                    indices_dset = file["layers"][x]["indices"]
                    indptr_dset = file["layers"][x]["indptr"]

                    name = f"layer {x} data"

                    (
                        selected_rows_data,
                        selected_rows_indices,
                        selected_rows_indptr,
                    ) = sparse_grab_filtered_values(
                        rows_to_load, data_dset, indices_dset, indptr_dset, name
                    )

                    # Create csr_matrix directly from NumPy arrays
                    print(
                        "Constructing data into csr_matrix format:  \U0001F527",
                        flush=True,
                    )
                    layers[x] = csr_matrix(
                        (
                            selected_rows_data,
                            selected_rows_indices,
                            selected_rows_indptr,
                        ),
                        shape=(len(rows_to_load), num_columns),
                        dtype=file["layers"][x]["data"].dtype,
                    )
                    print("Construction complete \u2705")

            else:
                layers = {}
                for x in (
                    value for value in filter_layers if value in file["layers"].keys()
                ):
                    # Assign variables to query
                    data_dset = file["layers"][x]["data"]
                    indices_dset = file["layers"][x]["indices"]
                    indptr_dset = file["layers"][x]["indptr"]

                    name = f"layer {x} data"

                    (
                        selected_rows_data,
                        selected_rows_indices,
                        selected_rows_indptr,
                    ) = sparse_grab_filtered_values(
                        obs_rows_to_load, var_cols_to_load, data_dset, indices_dset, indptr_dset, name
                    )

                    # Create csr_matrix directly from NumPy arrays
                    print(
                        "Constructing data into csr_matrix format:  \U0001F527",
                        flush=True,
                    )
                    layers[x] = csr_matrix(
                        (
                            selected_rows_data,
                            selected_rows_indices,
                            selected_rows_indptr,
                        ),
                        shape=(len(obs_rows_to_load), len(var_cols_to_load)),
                        dtype=file["layers"][x]["data"].dtype,
                    )
                    print("Construction complete \u2705")

        else:
            layers = None

        if keep_obsm:
            if not filter_obsm:
                obsm = {
                    x: anndata._io.h5ad.read_elem(file["obsm"][x])[obs_rows_to_load]
                    for x in file["obsm"].keys()
                }
            else:
                obsm = {
                    x: anndata._io.h5ad.read_elem(file["obsm"][x])[obs_rows_to_load]
                    for x in filter_obsm
                    if x in file["obsm"].keys()
                }
        else:
            obsm = None

        if keep_obsp:
            if not filter_obsp:
                obsp = {
                    x: anndata._io.h5ad.read_elem(file["obsp"][x])[obs_rows_to_load][
                        :, obs_rows_to_load
                    ]
                    for x in file["obsp"].keys()
                }
            else:
                obsp = {
                    x: anndata._io.h5ad.read_elem(file["obsp"][x])[obs_rows_to_load][
                        :, obs_rows_to_load
                    ]
                    for x in filter_obsp
                    if x in file["obsp"].keys()
                }
        else:
            obsp = None

        if keep_varm:
            if not filter_varm:
                varm = {
                    x: anndata._io.h5ad.read_elem(file["varm"])[var_cols_to_load]
                    for x in file["varm"].keys()
                }
            else:
                varm = {
                    x: anndata._io.h5ad.read_elem(file["varm"][x])[var_cols_to_load]
                    for x in filter_varm
                    if x in file["varm"].keys()
                }
        else:
            varm = None

        if keep_varp:
            if not filter_varp:
                varp = {
                    x: anndata._io.h5ad.read_elem(file["varp"][x])[var_cols_to_load][
                        :, var_cols_to_load
                    ]
                    for x in file["varp"].keys()
                }
            else:
                varp = {
                    x: anndata._io.h5ad.read_elem(file["varp"][x])[var_cols_to_load][
                        :, var_cols_to_load
                    ]
                    for x in filter_varp
                    if x in file["varp"].keys()
                }
        else:
            varp = None

        if keep_uns:
            if not filter_uns:
                uns = anndata._io.h5ad.read_elem(file["uns"])
            else:
                uns = {
                    x: anndata._io.h5ad.read_elem(file["uns"][x])
                    for x in filter_uns
                    if x in file["uns"].keys()
                }
        else:
            uns = None

        adata = anndata.AnnData(
            X=subset_matrix,
            obs=obs_dataframe,
            var=var_dataframe,
            layers=layers,
            obsm=obsm,
            obsp=obsp,
            varm=varm,
            varp=varp,
            uns=uns,
        )
        
        return adata

In [ ]:
input_settings = {
    'data_dir' : data_dir,
    'obs_filter_dict' : {
                        'LVL0' : ['Haematopoeitic_lineage'],
                        'organ' : ['Yolk_sac ']
                        },
    'obs_additional_cols_keep' : [],
    'obs_filter_method' : 'intersection',
    'var_filter_dict' : {
                        'intersection' : ['1'],
                        'highly_variable_reason' : ['hvg'],
                        'highly_variable_intersection' : ['False']
                        },
    'var_additional_cols_keep' : [],
    'var_filter_method' : 'intersection',
    
    'filter_layers' : [],
    'filter_obsm' : [],
    'filter_obsp' : [],
    'filter_varm' : [],
    'filter_varp' : [],
    'filter_uns' : [],
    
    'keep_layers' : False,
    'keep_obsm' : False,
    'keep_obsp' : False,
    'keep_varm' : False,
    'keep_varp' : False,
    'keep_uns' : False,
    
}

adata = create_anndata_subset(**input_settings)

print("")
print("")
print("\033[1mSubset anndata object generated successfully\033[0m\n")
print("\033[1m" + "Anndata whole preview:" + "\033[0m")
display(adata)
print("")
print("")
print("\033[1mQuick view of the anndata object generated\033[0m\n")
print(f"Overall shape: {adata.shape}")
print(f"Min count: {adata.X.min()}")
print(f"Max count: {adata.X.max()}")
print("")
print("\033[1m" + "obs preview:" + "\033[0m")
display(adata.obs)
print("")
print("\033[1m" + "var preview:" + "\033[0m")
display(adata.var)
print("")